# Marcas de itens homologados

Explora `tasks/api-compras/outputs/tabela-marcas.csv` e testa, em tabela temporária, o join das marcas com `item_homologado`.

In [9]:
import os
import subprocess
from pathlib import Path

import pandas as pd


def localizar_raiz_repositorio():
    for caminho in (Path.cwd(), *Path.cwd().parents):
        if (caminho / "tasks" / "alteracoes-no-banco-de-dados" / "tabela-marcas-compras-gov").is_dir():
            return caminho
    raise FileNotFoundError("Execute o notebook dentro do repositório cesta-de-precos-pncp.")


RAIZ_REPOSITORIO = localizar_raiz_repositorio()
CSV_PATH = RAIZ_REPOSITORIO / "tasks/api-compras/outputs/tabela-marcas.csv"
CSV_PATH

WindowsPath('c:/Users/rdurl/OneDrive/Documentos/cesta-de-precos-pncp/tasks/api-compras/outputs/tabela-marcas.csv')

## Leitura e perfil do CSV

In [10]:
marcas_df = pd.read_csv(CSV_PATH, dtype=str, keep_default_na=False)

pd.DataFrame({
    "linhas": [len(marcas_df)],
    "colunas": [len(marcas_df.columns)],
})

,linhas,colunas
0,65976,16


In [11]:
resumo_colunas = pd.DataFrame({
    "coluna": marcas_df.columns,
    "vazios_ou_na": [marcas_df[col].str.strip().isin(["", "NA"]).sum() for col in marcas_df.columns],
    "max_caracteres": [marcas_df[col].str.len().max() for col in marcas_df.columns],
})

resumo_colunas

,coluna,vazios_ou_na,max_caracteres
0,numero_controle_pncp,0,28
1,numero_item,0,2
2,ni_fornecedor,0,22
3,codigo_item_catalogo,0,6
4,id_compra,0,17
5,id_compra_item,0,22
6,codigo_item_catalogo_tbl_itens,1418,6
7,codigo_item_catalogo_tbl_marcas,0,6
8,quantidade_homologada,0,9
9,valor_unitario_homologado,0,11


In [12]:
chave_item = ["numero_controle_pncp", "numero_item"]
chave_linha = ["numero_controle_pncp", "numero_item", "ni_fornecedor", "id_compra_item", "marca"]

duplicidades_item = (
    marcas_df.groupby(chave_item, dropna=False)
    .size()
    .reset_index(name="linhas")
    .query("linhas > 1")
    .sort_values("linhas", ascending=False)
)

duplicidades_chave = (
    marcas_df.groupby(chave_linha, dropna=False)
    .size()
    .reset_index(name="linhas")
    .query("linhas > 1")
)

pd.DataFrame({
    "metrica": ["linhas", "itens_distintos", "itens_com_multiplas_linhas", "chaves_naturais_duplicadas", "marcas_distintas"],
    "valor": [
        len(marcas_df),
        marcas_df[chave_item].drop_duplicates().shape[0],
        len(duplicidades_item),
        len(duplicidades_chave),
        marcas_df["marca"].nunique(),
    ],
})

,metrica,valor
0,linhas,65976
1,itens_distintos,65976
2,itens_com_multiplas_linhas,0
3,chaves_naturais_duplicadas,0
4,marcas_distintas,22647


In [13]:
duplicidades_item.head(20)

,numero_controle_pncp,numero_item,linhas


## Join temporário e classificação incremental

A célula abaixo usa `psql` e variáveis de `.env`. Ela cria apenas tabelas temporárias, classifica linhas novas/existentes/conflitantes e termina com `ROLLBACK`.

In [ ]:
def carregar_env(caminho_env):
    if not caminho_env.exists():
        raise FileNotFoundError("Arquivo .env não encontrado na raiz do repositório.")

    valores = {}
    for linha in caminho_env.read_text(encoding="utf-8").splitlines():
        linha = linha.strip()
        if not linha or linha.startswith("#") or "=" not in linha:
            continue
        chave, valor = linha.split("=", 1)
        valores[chave.strip()] = valor.strip().strip('"').strip("'")
    return valores


def montar_env_psql():
    db_env = carregar_env(RAIZ_REPOSITORIO / ".env")
    obrigatorias = ["DB_HOST", "DB_PORT", "DB_USER", "DB_PASS"]
    ausentes = [chave for chave in obrigatorias if not db_env.get(chave)]
    if ausentes:
        raise KeyError(f"Variáveis ausentes no .env: {', '.join(ausentes)}")

    env = os.environ.copy()
    env.update({
        "PGHOST": db_env["DB_HOST"],
        "PGPORT": db_env["DB_PORT"],
        "PGUSER": db_env["DB_USER"],
        "PGPASSWORD": db_env["DB_PASS"],
        "PGDATABASE": "medicamentos_transparentes",
    })
    return env


def rodar_psql(sql):
    return subprocess.run(
        ["psql", "-v", "ON_ERROR_STOP=1"],
        input=sql,
        text=True,
        capture_output=True,
        env=montar_env_psql(),
        check=False,
    )

In [ ]:
CSV_PSQL = str(CSV_PATH).replace("\\", "/").replace("'", "''")

SQL_JOIN_TEST = rf"""
BEGIN;

CREATE TEMP TABLE tmp_item_homologado_marcas_csv (
  numero_controle_pncp TEXT,
  numero_item TEXT,
  ni_fornecedor TEXT,
  codigo_item_catalogo TEXT,
  id_compra TEXT,
  id_compra_item TEXT,
  codigo_item_catalogo_tbl_itens TEXT,
  codigo_item_catalogo_tbl_marcas TEXT,
  quantidade_homologada TEXT,
  valor_unitario_homologado TEXT,
  marca TEXT,
  descricao_detalhada_item TEXT,
  sigla_unidade_fornecimento TEXT,
  nome_unidade_fornecimento TEXT,
  sigla_unidade_medida TEXT,
  nome_unidade_medida TEXT
) ON COMMIT DROP;

\copy tmp_item_homologado_marcas_csv FROM '{CSV_PSQL}' WITH (FORMAT csv, HEADER true, NULL '')

CREATE TEMP TABLE tmp_item_homologado_marcas ON COMMIT DROP AS
SELECT
  btrim(numero_controle_pncp)::VARCHAR(30) AS numero_controle_pncp,
  btrim(numero_item)::INTEGER AS numero_item,
  btrim(ni_fornecedor)::VARCHAR(100) AS ni_fornecedor,
  btrim(codigo_item_catalogo)::INTEGER AS codigo_item_catalogo,
  btrim(id_compra)::VARCHAR(30) AS id_compra,
  btrim(id_compra_item)::VARCHAR(30) AS id_compra_item,
  CASE
    WHEN NULLIF(btrim(codigo_item_catalogo_tbl_itens), '') IS NULL
      OR upper(btrim(codigo_item_catalogo_tbl_itens)) = 'NA'
    THEN NULL
    ELSE btrim(codigo_item_catalogo_tbl_itens)::INTEGER
  END AS codigo_item_catalogo_tbl_itens,
  btrim(codigo_item_catalogo_tbl_marcas)::INTEGER AS codigo_item_catalogo_tbl_marcas,
  btrim(quantidade_homologada)::INTEGER AS quantidade_homologada,
  btrim(valor_unitario_homologado)::NUMERIC AS valor_unitario_homologado,
  btrim(marca)::VARCHAR(255) AS marca,
  CASE
    WHEN NULLIF(btrim(descricao_detalhada_item), '') IS NULL
      OR upper(btrim(descricao_detalhada_item)) = 'NA'
    THEN NULL
    ELSE btrim(descricao_detalhada_item)
  END AS descricao_detalhada_item,
  CASE
    WHEN NULLIF(btrim(sigla_unidade_fornecimento), '') IS NULL
      OR upper(btrim(sigla_unidade_fornecimento)) = 'NA'
    THEN NULL
    ELSE btrim(sigla_unidade_fornecimento)::VARCHAR(50)
  END AS sigla_unidade_fornecimento,
  CASE
    WHEN NULLIF(btrim(nome_unidade_fornecimento), '') IS NULL
      OR upper(btrim(nome_unidade_fornecimento)) = 'NA'
    THEN NULL
    ELSE btrim(nome_unidade_fornecimento)::VARCHAR(100)
  END AS nome_unidade_fornecimento,
  CASE
    WHEN NULLIF(btrim(sigla_unidade_medida), '') IS NULL
      OR upper(btrim(sigla_unidade_medida)) = 'NA'
    THEN NULL
    ELSE btrim(sigla_unidade_medida)::VARCHAR(50)
  END AS sigla_unidade_medida,
  CASE
    WHEN NULLIF(btrim(nome_unidade_medida), '') IS NULL
      OR upper(btrim(nome_unidade_medida)) = 'NA'
    THEN NULL
    ELSE btrim(nome_unidade_medida)::VARCHAR(100)
  END AS nome_unidade_medida
FROM tmp_item_homologado_marcas_csv;

CREATE TEMP TABLE tmp_item_homologado_marcas_distintas ON COMMIT DROP AS
SELECT DISTINCT *
FROM tmp_item_homologado_marcas;

SELECT
  (SELECT COUNT(*) FROM tmp_item_homologado_marcas_csv) AS linhas_csv,
  (SELECT COUNT(*) FROM tmp_item_homologado_marcas_distintas) AS linhas_distintas,
  (
    (SELECT COUNT(*) FROM tmp_item_homologado_marcas)
    - (SELECT COUNT(*) FROM tmp_item_homologado_marcas_distintas)
  ) AS duplicatas_exatas_ignoradas,
  COUNT(DISTINCT (numero_controle_pncp, numero_item)) AS itens_distintos,
  COUNT(DISTINCT marca) AS marcas_distintas
FROM tmp_item_homologado_marcas_distintas;

SELECT
  COUNT(*) AS chaves_com_valores_divergentes
FROM (
  SELECT
    numero_controle_pncp,
    numero_item,
    ni_fornecedor,
    id_compra_item,
    marca
  FROM tmp_item_homologado_marcas_distintas
  GROUP BY
    numero_controle_pncp,
    numero_item,
    ni_fornecedor,
    id_compra_item,
    marca
  HAVING COUNT(*) > 1
) AS divergentes;

SELECT
  COUNT(*) AS linhas_distintas,
  COUNT(item.numero_controle_pncp) AS linhas_com_join,
  COUNT(*) - COUNT(item.numero_controle_pncp) AS linhas_sem_join
FROM tmp_item_homologado_marcas_distintas AS marcas
LEFT JOIN item_homologado AS item
  ON item.numero_controle_pncp = marcas.numero_controle_pncp
 AND item.numero_item = marcas.numero_item;

CREATE TEMP TABLE tmp_item_homologado_marcas_classificacao ON COMMIT DROP AS
SELECT
  marcas.*,
  CASE
    WHEN existente.numero_controle_pncp IS NULL THEN 'nova'
    WHEN marcas.codigo_item_catalogo IS DISTINCT FROM existente.codigo_item_catalogo
      OR marcas.id_compra IS DISTINCT FROM existente.id_compra
      OR marcas.codigo_item_catalogo_tbl_itens IS DISTINCT FROM existente.codigo_item_catalogo_tbl_itens
      OR marcas.codigo_item_catalogo_tbl_marcas IS DISTINCT FROM existente.codigo_item_catalogo_tbl_marcas
      OR marcas.quantidade_homologada IS DISTINCT FROM existente.quantidade_homologada
      OR marcas.valor_unitario_homologado IS DISTINCT FROM existente.valor_unitario_homologado
      OR marcas.descricao_detalhada_item IS DISTINCT FROM existente.descricao_detalhada_item
      OR marcas.sigla_unidade_fornecimento IS DISTINCT FROM existente.sigla_unidade_fornecimento
      OR marcas.nome_unidade_fornecimento IS DISTINCT FROM existente.nome_unidade_fornecimento
      OR marcas.sigla_unidade_medida IS DISTINCT FROM existente.sigla_unidade_medida
      OR marcas.nome_unidade_medida IS DISTINCT FROM existente.nome_unidade_medida
    THEN 'conflito_divergente'
    ELSE 'existente_igual'
  END AS status_incremental
FROM tmp_item_homologado_marcas_distintas AS marcas
LEFT JOIN item_homologado_marcas AS existente
  ON existente.numero_controle_pncp = marcas.numero_controle_pncp
 AND existente.numero_item = marcas.numero_item
 AND existente.ni_fornecedor = marcas.ni_fornecedor
 AND existente.id_compra_item = marcas.id_compra_item
 AND existente.marca = marcas.marca;

SELECT
  status_incremental,
  COUNT(*) AS linhas
FROM tmp_item_homologado_marcas_classificacao
GROUP BY status_incremental
ORDER BY status_incremental;

SELECT
  numero_controle_pncp,
  numero_item,
  ni_fornecedor,
  id_compra_item,
  marca
FROM tmp_item_homologado_marcas_classificacao
WHERE status_incremental = 'conflito_divergente'
ORDER BY numero_controle_pncp, numero_item, id_compra_item, marca
LIMIT 20;

SELECT
  numero_controle_pncp,
  numero_item,
  marca,
  id_compra_item,
  status_incremental
FROM tmp_item_homologado_marcas_classificacao
ORDER BY numero_controle_pncp, numero_item, marca
LIMIT 20;

SELECT
  marcas.numero_controle_pncp,
  marcas.numero_item,
  marcas.marca,
  marcas.status_incremental,
  item.codigo_item_catalogo AS codigo_item_catalogo_item_homologado,
  marcas.codigo_item_catalogo AS codigo_item_catalogo_marcas
FROM tmp_item_homologado_marcas_classificacao AS marcas
INNER JOIN item_homologado AS item
  ON item.numero_controle_pncp = marcas.numero_controle_pncp
 AND item.numero_item = marcas.numero_item
ORDER BY marcas.numero_controle_pncp, marcas.numero_item, marcas.marca
LIMIT 20;

ROLLBACK;
"""

resultado = rodar_psql(SQL_JOIN_TEST)
print(resultado.stdout)
if resultado.stderr:
    print(resultado.stderr)